<a href="https://colab.research.google.com/github/dnhshl/cc-ai/blob/main/vectorDB.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 🚀 Einführung in Vektordatenbanken
Willkommen zu dieser Lehreinheit über Vektordatenbanken! Nachdem wir im vorherigen Notebook gelernt haben, wie man Text (oder andere Daten) in aussagekräftige numerische Vektoren (Embeddings) umwandelt, stellt sich die Frage: Wie können wir diese Vektoren effizient speichern, verwalten und vor allem durchsuchen?

Hier kommen Vektordatenbanken ins Spiel. Sie sind spezialisierte Datenbanksysteme, die für die Speicherung, Indizierung und Abfrage von hochdimensionalen Vektor-Embeddings optimiert sind.

**Was du in diesem Notebook lernen wirst:**
* Was Vektordatenbanken sind und warum sie im Zeitalter von KI so wichtig sind.
* Die Kernkonzepte hinter Vektordatenbanken (Ähnlichkeitssuche, Indizierung).
* Typische Anwendungsfälle für Vektordatenbanken.
* Ein praktisches Beispiel mit FAISS (Facebook AI Similarity Search), einer Bibliothek für effiziente Ähnlichkeitssuche, direkt in Colab.
* Einen kurzen Überblick über populäre Vektordatenbank-Lösungen.

**Voraussetzungen:**
* Grundlegendes Verständnis von Embeddings (siehe vorheriges Notebook).
* Python-Kenntnisse.
* Ein Google-Konto für Google Colab.

---

### 🛠️ 1. Vorbereitung: Installation notwendiger Bibliotheken
Wir benötigen `sentence-transformers`, um wieder Beispiel-Embeddings zu generieren, und `faiss-cpu` für unser praktisches Beispiel. FAISS gibt es auch als `faiss-gpu`, aber für den Einstieg und die Kompatibilität in Colab ohne spezielle GPU-Zuweisung ist die CPU-Version ausreichend.

* `sentence-transformers`: Zum Erzeugen von Text-Embeddings.
* `faiss-cpu`: Bibliothek für Ähnlichkeitssuche.
* `numpy`: Für numerische Operationen.

In [ ]:
!pip install sentence-transformers faiss-cpu numpy -q

print("✅ Bibliotheken erfolgreich installiert!")

---
### 🤔 2. Warum Vektordatenbanken?

Stell dir vor, du hast Millionen oder sogar Milliarden von Embeddings. Wie findest du schnell die Vektoren, die einem neuen Anfragevektor am ähnlichsten sind? Herkömmliche Datenbanken sind für diese Art von Abfragen (basierend auf Vektorähnlichkeit in hochdimensionalen Räumen) nicht optimiert.

**Traditionelle Datenbanken vs. Vektordatenbanken:**
* **Traditionelle Datenbanken** (SQL, NoSQL) sind gut darin, strukturierte oder semi-strukturierte Daten anhand exakter Übereinstimmungen oder definierter Beziehungen zu filtern (z.B. `SELECT * FROM users WHERE age > 30`).
* **Vektordatenbanken** sind darauf spezialisiert, die "nächsten Nachbarn" zu einem gegebenen Vektor in einem hochdimensionalen Raum zu finden. Dies wird oft als **Ähnlichkeitssuche** oder **Nearest Neighbor Search (NNS)** bezeichnet.

**Die Herausforderung der Dimensionalität:**
Embeddings haben oft hunderte oder tausende Dimensionen. In solch hochdimensionalen Räumen versagen traditionelle Indizierungsstrukturen (wie B-Bäume). Die Suche nach exakten nächsten Nachbarn (Exact Nearest Neighbor, ENN) wird extrem rechenaufwendig (oft als "Fluch der Dimensionalität" bezeichnet).

**Die Lösung: Approximate Nearest Neighbor (ANN) Search**
Die meisten Vektordatenbanken verwenden Algorithmen für die **ungefähre Ähnlichkeitssuche (Approximate Nearest Neighbor, ANN)**. Diese Algorithmen opfern ein klein wenig Genauigkeit, um eine enorme Geschwindigkeitssteigerung bei der Suche zu erzielen. Für viele Anwendungen ist eine "fast perfekte" Übereinstimmung ausreichend.

---

### 💡 3. Kernkonzepte von Vektordatenbanken

1.  **Vektor-Embeddings als Datenpunkte:**
    Das Herzstück jeder Vektordatenbank sind die Embeddings. Jedes Objekt (Text, Bild, Produkt, Nutzerprofil etc.) wird durch sein Embedding repräsentiert.

2.  **Distanzmetriken:**
    Um Ähnlichkeit zu messen, benötigen wir Distanz- oder Ähnlichkeitsmetriken. Gängige Metriken sind:
    * **Kosinus-Ähnlichkeit (Cosine Similarity):** Misst den Winkel zwischen zwei Vektoren. Werte reichen von -1 (entgegengesetzt) bis 1 (identisch). Oft wird die Kosinus-Distanz (1 - Kosinus-Ähnlichkeit) verwendet.
    * **Euklidische Distanz (L2-Distanz):** Der geradlinige Abstand zwischen zwei Punkten im Vektorraum.
    * **Skalarprodukt (Dot Product / Inner Product):** Kann auch als Ähnlichkeitsmaß verwendet werden, besonders wenn Vektoren normalisiert sind.

3.  **Indizierung für ANN Search:**
    Um die ANN-Suche zu beschleunigen, erstellen Vektordatenbanken spezielle Indexstrukturen. Diese Strukturen organisieren die Vektoren so, dass der Suchraum effizient eingeschränkt werden kann. Bekannte Indizierungsalgorithmen sind:
    * **Flat:** Keine Indizierung, brute-force Suche. Nur für sehr kleine Datensätze geeignet.
    * **LSH (Locality Sensitive Hashing):** Gruppiert ähnliche Vektoren mithilfe von Hash-Funktionen.
    * **Tree-based (z.B. Annoy):** Baut Baumstrukturen auf, um den Raum zu partitionieren.
    * **Clustering-based (z.B. IVFADC in FAISS):** Gruppiert Vektoren in Cluster (z.B. mit K-Means) und durchsucht nur relevante Cluster. Oft kombiniert mit Quantisierung zur Komprimierung der Vektoren.
    * **Graph-based (z.B. HNSW):** Konstruiert einen Graphen, in dem Knoten Vektoren und Kanten Ähnlichkeitsbeziehungen darstellen. Die Suche erfolgt durch Traversieren des Graphen.

4.  **Abfragemechanismen:**
    Die typische Abfrage an eine Vektordatenbank ist: "Finde die Top-K Vektoren, die diesem Anfragevektor am ähnlichsten sind."

---

### 🎯 4. Anwendungsfälle

Vektordatenbanken ermöglichen eine Vielzahl von KI-gestützten Anwendungen:

* **Semantische Suche:** Suche nach Dokumenten, Produkten oder Bildern basierend auf ihrer Bedeutung, nicht nur auf Keywords (z.B. eine Suchmaschine, die "gesunde Snacks für Kinder" versteht, auch wenn die Produkte nicht genau diese Wörter enthalten).
* **Empfehlungssysteme:** Empfehle Nutzern ähnliche Artikel, Lieder, Filme oder sogar andere Nutzer basierend auf Ähnlichkeiten ihrer Embeddings.
* **Frage-Antwort-Systeme (Q&A) & Retrieval Augmented Generation (RAG):** Finde relevante Textpassagen in einer Wissensdatenbank, um Fragen zu beantworten oder LLMs (Large Language Models) mit externem Wissen zu versorgen.
* **Anomalieerkennung:** Identifiziere Datenpunkte, die sich signifikant von anderen unterscheiden (deren Embeddings weit von anderen entfernt liegen).
* **Bildähnlichkeitssuche (Reverse Image Search):** Finde ähnliche Bilder zu einem gegebenen Bild.
* **Deduplizierung:** Finde und entferne doppelte oder sehr ähnliche Inhalte.
* **Clustering:** Gruppiere ähnliche Objekte automatisch.

---

### 💻 5. Praktisches Beispiel mit FAISS

FAISS (Facebook AI Similarity Search) ist eine Open-Source-Bibliothek, die effiziente Algorithmen für die Ähnlichkeitssuche und das Clustering von dichten Vektoren bietet. Es ist keine vollwertige Datenbank mit API-Endpunkten und Persistenz im traditionellen Sinne, aber es implementiert die Kernfunktionalität der Vektorindizierung und -suche und kann als Backend für solche Systeme dienen oder direkt für In-Memory-Anwendungen genutzt werden.

**Schritte:**
1.  Wir generieren einige Beispiel-Embeddings (wie im vorherigen Notebook).
2.  Wir erstellen einen einfachen FAISS-Index.
3.  Wir fügen unsere Embeddings zum Index hinzu.
4.  Wir führen eine Ähnlichkeitssuche durch.

In [ ]:
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

# 1. Beispiel-Embeddings generieren
model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

documents = [
    "Das Wetter ist heute schön sonnig.",
    "Ich genieße die warmen Sonnenstrahlen.",
    "Morgen wird es wahrscheinlich regnen.",
    "Künstliche Intelligenz revolutioniert unsere Welt.",
    "Viele KI-Modelle basieren auf neuronalen Netzen.",
    "Ein Apfel pro Tag ist gesund."
]

doc_embeddings = model.encode(documents)
doc_embeddings = np.array(doc_embeddings).astype('float32') # FAISS benötigt float32 und numpy array

print(f"Form der Dokument-Embeddings: {doc_embeddings.shape}") # (Anzahl Dokumente, Embedding-Dimension)
embedding_dim = doc_embeddings.shape[1]

# 2. Einen FAISS-Index erstellen
# IndexFlatL2: Einfacher Index, der die exakte euklidische Distanz berechnet.
# Für Kosinus-Ähnlichkeit normalisieren wir die Vektoren und verwenden IndexFlatIP (Inner Product).

# Normalisieren der Vektoren für Kosinus-Ähnlichkeit mit Skalarprodukt
faiss.normalize_L2(doc_embeddings)

# index = faiss.IndexFlatL2(embedding_dim) # Für Euklidische Distanz
index = faiss.IndexFlatIP(embedding_dim)    # Für Kosinus-Ähnlichkeit (nach Normalisierung)
print(f"Ist der Index trainiert? {index.is_trained}") # IndexFlat* benötigen kein Training

# 3. Embeddings zum Index hinzufügen
index.add(doc_embeddings)
print(f"Anzahl der Vektoren im Index: {index.ntotal}")

# 4. Ähnlichkeitssuche durchführen
query_text = "Wie ist das Wetter?"
query_embedding = model.encode([query_text])
query_embedding = np.array(query_embedding).astype('float32')
faiss.normalize_L2(query_embedding) # Anfragevektor ebenfalls normalisieren

k = 3 # Anzahl der ähnlichsten Ergebnisse, die wir wollen
D, I = index.search(query_embedding, k) # D = Distanzen/Ähnlichkeiten, I = Indizes der Ergebnisse

print(f"\nSuche für: '{query_text}'")
print(f"Top {k} Ergebnisse:")
for i in range(k):
    doc_index = I[0][i]
    similarity_score = D[0][i]
    print(f"  - Dokument {doc_index + 1}: '{documents[doc_index]}' (Ähnlichkeit: {similarity_score:.4f})")

query_text_2 = "Was sind die neuesten Entwicklungen im Bereich KI?"
query_embedding_2 = model.encode([query_text_2])
query_embedding_2 = np.array(query_embedding_2).astype('float32')
faiss.normalize_L2(query_embedding_2)

D2, I2 = index.search(query_embedding_2, k)
print(f"\nSuche für: '{query_text_2}'")
print(f"Top {k} Ergebnisse:")
for i in range(k):
    doc_index = I2[0][i]
    similarity_score = D2[0][i]
    print(f"  - Dokument {doc_index + 1}: '{documents[doc_index]}' (Ähnlichkeit: {similarity_score:.4f})")

**Erläuterung zum FAISS-Beispiel:**
* Wir haben `IndexFlatIP` verwendet, was für "Index Flat Inner Product" steht. Nachdem wir die Vektoren L2-normalisiert haben, entspricht das maximale Skalarprodukt der maximalen Kosinus-Ähnlichkeit.
* `index.search(query_embedding, k)` gibt zwei Arrays zurück:
    * `D`: Enthält die Distanzen (oder Ähnlichkeits-Scores, je nach Index) der `k` nächsten Nachbarn.
    * `I`: Enthält die Indizes (Positionen in der ursprünglichen Hinzufügereihenfolge) dieser `k` nächsten Nachbarn.
* Für größere Datensätze würde man komplexere FAISS-Indizes verwenden (z.B. `IndexIVFFlat`, `IndexHNSWFlat`), die ein Training erfordern und Kompromisse zwischen Suchgeschwindigkeit, Genauigkeit und Speicherbedarf bieten.

---
### 🌐 6. Überblick über populäre Vektordatenbanken

Neben FAISS, das eine Bibliothek ist, gibt es viele dedizierte Vektordatenbank-Systeme. Hier eine Auswahl (viele bieten kostenlose Testversionen oder Open-Source-Optionen):

* **Pinecone:** Eine vollständig verwaltete Cloud-Vektordatenbank. Bekannt für einfache Bedienung und Skalierbarkeit. Bietet einen kostenlosen Starter-Plan.
* **Weaviate:** Eine Open-Source-Vektordatenbank mit GraphQL-API. Kann selbst gehostet (z.B. mit Docker) oder als verwalteter Dienst genutzt werden. Bietet auch semantische Suchfunktionen über Keywords hinaus.
* **Milvus:** Eine Open-Source-Vektordatenbank, die für hohe Leistung und Skalierbarkeit entwickelt wurde. Unterstützt verschiedene ANN-Indizes und Distanzmetriken.
* **ChromaDB:** Eine Open-Source-Vektordatenbank, die sich auf Einfachheit und Entwicklerfreundlichkeit konzentriert. Gut für den Start und kann In-Memory, als Client-Server oder persistent betrieben werden.
* **Qdrant:** Eine Open-Source-Vektordatenbank, geschrieben in Rust, mit Fokus auf Leistung und Filterfunktionen neben der Vektorsuche.
* **Elasticsearch / OpenSearch:** Diese bekannten Suchmaschinen bieten mittlerweile auch leistungsfähige Funktionen für die Vektorähnlichkeitssuche (Dense Vector Search).
* **Cloud-spezifische Lösungen:**
    * **Google Vertex AI Vector Search (Matching Engine):** Ein vollständig verwalteter Dienst auf Google Cloud.
    * **Amazon OpenSearch Service (mit k-NN Plugin):** Bietet Vektorsuchfunktionen.
    * **Azure AI Search (Semantic Search):** Integriert Vektorsuche und semantische Ranking-Funktionen.

Die Wahl hängt von Faktoren wie Datenvolumen, Abfrageanforderungen, Budget, gewünschtem Verwaltungsaufwand und spezifischen Features ab.

---

### 🤔 7. Wie wählt man eine Vektordatenbank aus?

Bei der Auswahl einer Vektordatenbank solltest du folgende Kriterien berücksichtigen:

1.  **Skalierbarkeit:** Kann die Datenbank mit der wachsenden Anzahl deiner Vektoren und Abfragen umgehen?
2.  **Leistung:** Wie schnell sind Indexierung und Abfrage (Latenz, Durchsatz)? Wie gut ist die Genauigkeit der ANN-Suche (Recall)?
3.  **Kosten:** Open Source vs. kommerziell, Self-Hosted vs. Managed Service. Berücksichtige auch Infrastrukturkosten.
4.  **Benutzerfreundlichkeit:** Wie einfach ist die Einrichtung, Integration und Wartung? Gibt es gute SDKs und Dokumentation?
5.  **Features:**
    * Unterstützte Distanzmetriken und Index-Typen.
    * Möglichkeit, Metadaten zusammen mit Vektoren zu speichern und zu filtern (hybride Suche).
    * CRUD-Operationen (Create, Read, Update, Delete) für Vektoren.
    * Sicherheitsfeatures, Backup-Möglichkeiten.
    * Echtzeit-Updates des Index.
6.  **Ökosystem und Community:** Wie aktiv ist die Community? Gibt es gute Integrationen mit anderen Tools (z.B. LLM-Frameworks wie LangChain, LlamaIndex)?

---

### 🎉 8. Zusammenfassung und nächste Schritte

**Herzlichen Glückwunsch!** Du hast nun einen Einblick in die Welt der Vektordatenbanken erhalten.

**Was haben wir gelernt?**
* Vektordatenbanken sind essentiell, um Embeddings effizient zu speichern und Ähnlichkeitssuchen durchzuführen.
* Sie nutzen ANN-Algorithmen und spezielle Indizes, um auch bei Milliarden von Vektoren schnelle Ergebnisse zu liefern.
* Die Anwendungsfälle reichen von semantischer Suche über Empfehlungen bis hin zu RAG für LLMs.
* Mit FAISS konnten wir die Kernfunktionalität der Ähnlichkeitssuche praktisch in Colab ausprobieren.
* Es gibt eine wachsende Landschaft an Open-Source- und kommerziellen Vektordatenbank-Lösungen.

**Mögliche nächste Schritte:**
* **Experimentiere mit FAISS:** Probiere verschiedene Index-Typen in FAISS aus (z.B. `IndexIVFFlat`). Beachte, dass einige Indizes ein `train()`-Schritt auf einem Teil der Daten benötigen, bevor Vektoren hinzugefügt werden können.
* **Teste eine gehostete Vektordatenbank:** Registriere dich für einen kostenlosen Tier bei Pinecone oder einem anderen Anbieter und repliziere das Beispiel.
* **Integriere mit LLM-Frameworks:** Schau dir an, wie LangChain oder LlamaIndex Vektordatenbanken für RAG-Anwendungen nutzen.
* **Hybride Suche:** Erforsche, wie man Metadaten-Filterung mit Vektorähnlichkeitssuche kombiniert.

Vektordatenbanken sind ein spannendes und sich schnell entwickelndes Feld. Viel Spaß beim weiteren Entdecken! 🌌